# Lesson 3: Agentic Search

In [ ]:
import subprocess
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)

_here = Path.cwd()
_req = _here / "requirements-lesson-03.txt"
if not _req.exists():
    _req = _here.parent / "requirements-lesson-03.txt"

if _req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Installed from", _req)
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "python-dotenv", "tavily-python", "requests", "beautifulsoup4",
        "duckduckgo-search", "pygments", "ipykernel",
    ])
    print("Installed packages via pip (requirements file not found).")

In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]

'The Blackwell GPU features fifth-generation NVLink, Transformer Engine, and HBM3E memory. It supports up to 576 GPUs and accelerates generative AI. It achieves up to 15 petaFLOPS performance.'

## Regular search

Choose location (try to change to your own city!).

In [2]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

**Note:** search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [6]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [  # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results


for i in search(query):
    print(i)

/var/folders/2p/x0g54f5s6_v5t47f_25hgyww0000gq/T/ipykernel_22970/914268248.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


https://weather.com/weather/today/l/San+Francisco+CA?canonicalCityId=d060bcef6a904af38313393bd51e4c3c
https://www.localconditions.com/us/san-francisco/california/weather/
https://www.theweathernetwork.com/en/city/us/california/san-francisco/current
https://www.weather-forecast.com/locations/San-Francisco/forecasts/latest
https://www.weather.gov/mtr/
https://weather.yahoo.com/us/ca/san-francisco/


In [8]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."

    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

**Note:** This produces a long output — you may want to right-click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [9]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000])  # limit long outputs

Website: https://weather.com/weather/today/l/San+Francisco+CA?canonicalCityId=d060bcef6a904af38313393bd51e4c3c


<body class="inter_1db00677-module__WhbPaa__className font-sans"><div hidden=""><!--$--><!--/$--></div><script>(self.__next_s=self.__next_s||[]).push(["/wx-next-web/static/vendor/@mparticle/web-sdk@2.47.1/dist/mparticle.min.js",{}])</script><script>(self.__next_s=self.__next_s||[]).push(["https://weather.com/api/v1/script/dprSdkScript.js",{"id":"dprsdk-script"}])</script><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"\n\t\t\t\t\tif (window.top?.DprSdk) {\n\t\t\t\t\t\tasync function initDprSdk() {\n\t\t\t\t\t\t\ttry {\n\t\t\t\t\t\t\t\tawait window.top?.DprSdk.init({\n\t\t\t\t\t\t\t\t\tgetApplicationInfo: () => ({ id: 'weather.com', version: '2.0.0' }),\n\t\t\t\t\t\t\t\t\tgetUserRegime: () => 'usa-ccpa',\n\t\t\t\t\t\t\t\t});\n\t\t\t\t\t\t\t} catch (_error) {\n\t\t\t\t\t\t\t\t// do nothing.\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\t}\n\t\t\t\t\t\tinitDprSdk();\n\t\t\t\t\t}\

In [10]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/San+Francisco+CA?canonicalCityId=d060bcef6a904af38313393bd51e4c3c


Home Home Today Today Hourly Hourly 10 Day 10 Day Monthly Monthly Radar Radar Video Video Home Home Today Today Hourly Hourly 10 Day 10 Day Monthly Monthly Radar Radar Video Video San Francisco, California Weather Wind Humidity Air Quality Dew Point Pressure UV Index Visibility Moon Phase Sunrise Sunset Hourly Weather Trending Now The Heat Dome That Refuses To Budge: March Heat Wave Breaks Records Slow Moving Wall Of Snow And Ice Hits Turkey 0:41 Severe Storms To Rumble Across Midwest, Ohio Valley Thursday 0:42 No Relief From The Weather This Week For Allergy Sufferers 0:40 The Heat Dome That Refuses To Budge: March Heat Wave Breaks Records Slow Moving Wall Of Snow And Ice Hits Turkey 0:41 Severe Storms To Rumble Across Midwest, Ohio Valley Thursday 0:42 No Relief From The Weather This Week For Allergy Sufferers 0:40 Daily Forecast Travel And The Best State For An Adventure 

## Agentic Search

In [11]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1774398508, 'localtime': '2026-03-24 17:28'}, 'current': {'last_updated_epoch': 1774397700, 'last_updated': '2026-03-24 17:15', 'temp_c': 23.3, 'temp_f': 73.9, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 10.1, 'wind_kph': 16.2, 'wind_degree': 282, 'wind_dir': 'WNW', 'pressure_mb': 1017.0, 'pressure_in': 30.03, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 36, 'cloud': 25, 'feelslike_c': 24.1, 'feelslike_f': 75.5, 'windchill_c': 21.7, 'windchill_f': 71.1, 'heatindex_c': 23.0, 'heatindex_f': 73.4, 'dewpoint_c': 4.4, 'dewpoint_f': 39.9, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 1.8, 'gust_mph': 18.9, 'gust_kph': 30.5}}


In [12]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)

{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1774398508,
        "localtime": "2026-03-24 17:28"
    },
    "current": {
        "last_updated_epoch": 1774397700,
        "last_updated": "2026-03-24 17:15",
        "temp_c": 23.3,
        "temp_f": 73.9,
        "is_day": 1,
        "condition": {
            "text": "Sunny",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
            "code": 1000
        },
        "wind_mph": 10.1,
        "wind_kph": 16.2,
        "wind_degree": 282,
        "wind_dir": "WNW",
        "pressure_mb": 1017.0,
        "pressure_in": 30.03,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 36,
        "cloud": 25,
        "feelslike_c": 24.1,
        "feelslike_f": 75.5,
        "windchill_c": 21.7,
        "wi